# Agents

In [1]:
# @title Installs
############################################

# Google Cloud Vertex AI (Agent Engine) & Google ADK installieren
# Feste Versionen statt --upgrade: Der Deploy pinnt dieselben Versionen, damit das Pickle in der Cloud passt
%pip install --quiet "google-cloud-aiplatform[agent_engines,adk]==2.6.0" "google-adk==2.12.0" "google-genai==2.33.0"

# Versionen kurz prüfen
%pip show google-adk google-cloud-aiplatform

Note: you may need to restart the kernel to use updated packages.
Name: google-adk
Version: 2.12.0
Summary: Agent Development Kit
Home-page: https://google.github.io/adk-docs/
Author: 
Author-email: Google LLC <googleapis-packages@google.com>
License: 
Location: /opt/anaconda3/lib/python3.12/site-packages
Requires: aiosqlite, authlib, click, cryptography, fastapi, google-auth, google-genai, graphviz, httpx, jsonschema, opentelemetry-api, opentelemetry-sdk, packaging, pydantic, python-dotenv, python-multipart, pyyaml, requests, starlette, tenacity, typing-extensions, uvicorn, watchdog, websockets
Required-by: 
---
Name: google-cloud-aiplatform
Version: 2.6.0
Summary: Vertex AI API client library
Home-page: https://github.com/googleapis/python-aiplatform
Author: Google LLC
Author-email: googleapis-packages@google.com
License: Apache 2.0
Location: /opt/anaconda3/lib/python3.12/site-packages
Requires: certifi, docstring_parser, google-api-core, google-auth, google-cloud-bigquery, google-cl

In [1]:
# @title Imports
############################################

import logging
import os
import re
import warnings
from importlib.metadata import version

import vertexai
from google.adk.agents import LlmAgent
from google.adk.agents.context import Context
from google.adk.events.event import Event
from google.adk.tools import VertexAiSearchTool, google_search, load_memory, url_context
from google.adk.workflow import JoinNode, RetryConfig, Workflow, node
from vertexai.preview import reasoning_engines

warnings.filterwarnings('ignore')

print("✅ Alle Imports (inkl. ADK 2.12 Workflow API) erfolgreich geladen!")

✅ Alle Imports (inkl. ADK 2.12 Workflow API) erfolgreich geladen!


In [ ]:
# @title Environmental Variables & GCP Setup
############################################

PROJECT_ID = "deltorobarba"
LOCATION = "us-central1"  # Agent Engine Region

# These tell the underlying google-genai client to use Vertex AI (Gemini 3.8 Flash uses 'global')
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

# Initialize Vertex AI SDK & Client
vertexai.init(project=PROJECT_ID, location=LOCATION)
client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

print("✅ Erfolgreich über Application Default Credentials (ADC) verbunden.")

**Workflow-Graph unserer Pipeline**

```mermaid
%%{init: {'themeVariables': {'fontSize': '13px'}, 'flowchart': {'nodeSpacing': 20, 'rankSpacing': 28, 'padding': 6}}}%%
flowchart TD
    START(["User Query"]) --> web_agent["1a. web_researcher<br/>google_search"]
    START --> doc_agent["1b. internal_specialist<br/>Vertex AI Search"]
    doc_agent --> retry_gate{{"2. retry_gate"}}
    retry_gate -->|"retry, max. 3×"| doc_agent
    web_agent --> results_join["3. results_join"]
    retry_gate -->|"ok"| results_join
    results_join --> clean_search_data["4. clean_search_data"]
    clean_search_data --> search_analyst["5. web_analyst<br/>url_context"]
    search_analyst --> analyst["6. analyst<br/>load_memory"]
    analyst --> summarizer["7. summarizer"]
```

1. **Parallel Fan-Out (`("START", (web_agent, doc_agent))`):** Sobald eine Nutzeranfrage eingeht, starten `web_researcher` (öffentliche Google-Suche) und `internal_specialist` (interne RAG-Suche über Vertex AI Search) gleichzeitig.
2. **Bedingte Korrekturschleife (`retry_gate` mit `route="retry"` / `route="ok"`):** Eine reine Python-Funktion (`@node`) prüft nach jedem Lauf von `internal_specialist`, ob Vertex AI Search Ergebnisse geliefert hat (`internal_data`). Falls `"NO INTERNAL DOCUMENTS FOUND"` oder ein leerer String zurückkam und noch weniger als **3 Versuche** stattgefunden haben, gibt `retry_gate` `Event(..., route="retry")` zurück und triggert `internal_specialist` erneut. Andernfalls wird der Retry-Zähler auf `0` zurückgesetzt und über `route="ok"` an `results_join` weitergeleitet.
3. **Fan-In Synchronisations-Barriere (`JoinNode`):** `results_join` wartet deterministisch, bis **beide** parallelen Pfade (`web_agent` **und** der `ok`-Pfad von `retry_gate`) abgeschlossen sind.
4. **Reiner Python-Transformationsschritt (`clean_search_data`):** Ohne einen zusätzlichen LLM-Call zu verschwenden, extrahiert diese `@node`-Funktion per Regex alle validen `https://`-URLs aus `{search_results}`, entfernt Duplikate (`dict.fromkeys`) und versieht `{internal_data}` mit einem Audit-Stempel (`[VAIS Retrieve Attempts: X/3]`).
5. **Sequenzielle Auswertung (`search_analyst` → `analyst` → `summarizer`):** `web_analyst` liest die bereinigten URLs via `url_context`, `analyst` gleicht öffentliche und interne Daten unter Einbezug der **Vertex AI Memory Bank** (`load_memory`) ab, und `summarizer` erstellt das Executive-Briefing.

**Wichtiger Hinweis zu `include_contents='default'` in ADK 2.12:** In `Workflow` setzt ADK für `LlmAgent`-Knoten standardmäßig `include_contents='none'` (sofern nicht angegeben), sodass ein Knoten nur seinen direkten `node_input` sieht. Durch `include_contents='default'` behalten die Agenten – wie zuvor im `SequentialAgent` – vollen Zugriff auf die bisherige Session-Historie (wichtig für Multi-Turn-Rückfragen).

*Notiz: Andere Modelle als Gemini (nicht Teil der Pipeline)*

ADK kennt zwei Wege, einem Agent ein Modell zu geben:

| Weg | Wann | Beispiel |
|---|---|---|
| **Name als Text** | Gemini auf Vertex AI | `model="gemini-3.8-flash"` (wie in dieser Pipeline) |
| **Wrapper-Objekt** | andere Anbieter und Modelle, z. B. über LiteLLM | `model=LiteLlm(model="vertex_ai/…")` |

**Beispiel: DeepSeek aus dem Model Garden** (am 2026-10-30 mit ADK 2.13 getestet: der Agent rief ein Funktions-Tool korrekt auf)

```python
from google.adk.models.lite_llm import LiteLlm

deepseek = LiteLlm(
    model="vertex_ai/deepseek-ai/deepseek-v4.1-flash-maas",  # Präfix vertex_ai/ + Modell-ID aus dem Model Garden
    vertex_project=PROJECT_ID,
    vertex_location="global",  # Model-Garden-Modelle (MaaS) laufen meist auf 'global'
    timeout=300,               # weitere Argumente gehen direkt an LiteLLM
)

agent = LlmAgent(
    name="deepseek_agent",
    model=deepseek,
    instruction="...",
    tools=[eine_python_funktion],  # nur eigene Funktions-Tools, siehe unten
)
```

Dasselbe Muster gilt für andere Modelle aus dem Model Garden, z. B. `vertex_ai/zai-org/glm-5.2-maas` (am 2026-10-30 erreichbar, aber 429 wegen Kontingent). Die Modell-IDs stehen in `models.ipynb`.

**Worauf man achten muss**

* **Import:** `from google.adk.models.lite_llm import LiteLlm`. Der Pfad `google.adk.agents.llm` aus älteren Notizen gibt einen `ImportError`.
* **Zusatzargumente:** `LiteLlm(model, **kwargs)` reicht alles direkt an LiteLLM weiter: `timeout=300`, nicht `completion_config={"timeout": 300}` (diesen Parameter gibt es nicht).
* **Gemini-eigene Tools nur mit Gemini:** `VertexAiSearchTool`, `google_search` und `url_context` laufen nur mit Gemini; ADK bricht sonst ab („Vertex AI search tool is not supported for model …“). Ein Agent mit DeepSeek braucht dafür ein eigenes Funktions-Tool, das z. B. die Vertex-AI-Search-API aufruft.
* **Kein `ollama/…` auf Agent Engine:** `ollama/` heißt ein lokaler Ollama-Server auf dem eigenen Rechner; den gibt es in der Cloud nicht. Modelle mit „base“ im Namen sind außerdem nicht auf Anweisungen trainiert und rufen keine Tools auf.
* **Region:** Das Modell muss nicht in derselben Region liegen wie die Engine, aber in der Region verfügbar sein, die man aufruft.
* **Anmeldung:** über ADC, kein API-Key. Deployed braucht die Identität des Agents `aiplatform.endpoints.predict`; bei Agent Identity steckt das schon in `agentDefaultAccess`.
* **Lokale Umgebung:** `litellm` 1.106 braucht `mcp` ab Version 2. In conda base ist `mcp` 1.30 installiert (für `mcp.ipynb` auf `<2` festgelegt), dort scheitert `LiteLlm` deshalb schon beim ersten Aufruf. Getestet wurde in einer eigenen Umgebung (ADK 2.13.0, litellm 1.106.6, mcp 2.3.0). Für einen Deploy gehören `litellm` und `mcp>=2` in die Requirements.

In [ ]:
# @title Define Agents and Workflow (ADK 2.12)
############################################

# Vertex AI Search (VAIS)
DATA_STORE_ID = "projects/deltorobarba/locations/global/collections/default_collection/dataStores/deltorobarba-adk_1773523070796"
# Google's out-of-the-box RAG system for information retrieval (for Grounding)
  # Documents are loaded onto Google Cloud Storage bucket (here: 'gs://deltorobarba_adk/')
  # Create new in 'Search app' (Custom search (general) in Vertex AI Search
  # --> Discovery Engine API is underlying engine, includes streamAssist. https://docs.cloud.google.com/generative-ai-app-builder/docs
  # VAIS does (managed): Chunk documents. Choose embedding model. Upload vectors to database. Search that database. Re-rank results.
# https://developers.googleblog.com/building-with-gemini-embedding-2/

# Automatic backoff retry for transient 429 rate-limits on parallel LLM calls (up to 4 retries, max. 60 s apart)
llm_retry = RetryConfig(max_attempts=5, initial_delay=5.0, max_delay=60.0, backoff_factor=2.0)

# 1a. Google Search Agent
web_agent = LlmAgent(
    name='web_researcher',
    model='gemini-3.8-flash',
    description=('Finds the latest public news about Gemini via Google Search.'),
    instruction='Search for the latest public news on the topic. Return only a list of the source URLs, one per line.',
    tools=[google_search],
    include_contents='default',
    output_key='search_results',
    retry_config=llm_retry,
)

# 1b. Vertex AI Search Agent
doc_agent = LlmAgent(
    name='internal_specialist',
    model='gemini-3.8-flash',
    instruction='''Search our internal knowledge base for related documents.
    If the tool returns no results, output EXACTLY: "NO INTERNAL DOCUMENTS FOUND".''',
    tools=[VertexAiSearchTool(data_store_id=DATA_STORE_ID)],  # Vertex AI Search with DATA_STORE_ID
    include_contents='default',
    output_key='internal_data',
    retry_config=llm_retry,
)

# 2. Conditional Retry Gate (@node) — Max 3 attempts for Vertex AI Search
@node
def retry_gate(ctx: Context, node_input) -> Event:
    attempts = ctx.state.get('internal_retries', 0) + 1
    internal_text = str(ctx.state.get('internal_data', '')).strip()

    if ('NO INTERNAL DOCUMENTS FOUND' in internal_text or not internal_text) and attempts < 3:
        print(f'⚠️ [retry_gate] Versuch {attempts}/3 ohne interne Dokumente -> Retry internal_specialist...')
        return Event(
            output='Please retry searching the internal knowledge base for related Gemini documents.',
            route='retry',
            state={'internal_retries': attempts},
        )

    # Erfolg oder max. 3 Versuche erreicht -> Zähler für Folgerunden zurücksetzen & weiterleiten
    return Event(
        output=internal_text,
        route='ok',
        state={'internal_retries': 0, 'vais_attempts_used': attempts},
    )

# 3. Fan-in Synchronization Barrier (JoinNode)
results_join = JoinNode(name='results_join')

# 4. Pure Python Transformation Step (@node) — URL Deduplication & Metadata Tagging
@node
def clean_search_data(ctx: Context, node_input) -> Event:
    raw_urls = str(ctx.state.get('search_results', ''))
    unique_urls = list(dict.fromkeys(re.findall(r'https?://[^\s,)\"<>]+', raw_urls)))
    cleaned_urls = '\n'.join(unique_urls) if unique_urls else raw_urls.strip()

    attempts_used = ctx.state.get('vais_attempts_used', 1)
    internal_data = f"[VAIS Retrieve Attempts: {attempts_used}/3]\n{ctx.state.get('internal_data', '')}"

    return Event(
        output=cleaned_urls,
        state={'search_results': cleaned_urls, 'internal_data': internal_data},
    )

# 5. Web Search Analysis Agent (Processes cleaned URLs)
search_analyst = LlmAgent(
    name='web_analyst',
    model='gemini-3.8-flash',
    instruction='''Use the url_context tool to read the URLs in {search_results}.
    Extract only technical specifications and performance benchmarks.''',
    tools=[url_context],
    include_contents='default',
    output_key='web_analysis',
    retry_config=llm_retry,
)

# 6. Deep Analysis Agent ("Brain")
analyst = LlmAgent(
    name='analyst',
    #model=deepseek,                   # <----- Be careful with region of 3P models for deployed version
    model='gemini-3.8-flash',          # hier eventually Gemini Pro
    description=('Compares Gemini 3 vs GPT-4o.'),
    instruction='''
    Check your memory for any user-specific instructions or past preferences.
    Compare public findings ({web_analysis}) against our private data ({internal_data}).
    Identify unique Gemini 3 features that GPT-4o lacks.
    ''',
    tools=[load_memory],  # Recall information from Vertex AI Memory Services!
    include_contents='default',
    output_key='final_comparison',
    retry_config=llm_retry,
)

# 7. CEO Summarizer Agent
summarizer = LlmAgent(
    name='summarizer',
    model='gemini-3.8-flash',
    description=('Draft concise summary'),
    instruction='''
    You are an Executive Intelligence Analyst.
    Summarize the findings in {final_comparison} into 5 punchy bullets for the CEO.
    Ensure you highlight contradictions between internal and public data.
    Max 2000 characters.
    ''',
    include_contents='default',
    output_key='summary',  # not used in deployed version
    retry_config=llm_retry,
)

# 8. Root Orchestrator (ADK 2.12 Graph Workflow)
root_agent = Workflow(
    name='researcher',
    edges=[
        # 1. Parallel Execution (Fan-out)
        ('START', (web_agent, doc_agent)),
        # 2. Conditional Loop (Max 3 retries for internal_specialist)
        (doc_agent, retry_gate),
        (retry_gate, {'retry': doc_agent, 'ok': results_join}),
        # 3. Fan-in Synchronization Barrier
        (web_agent, results_join),
        # 4. Pure Python Transformation + Sequential Analysis Chain
        (results_join, clean_search_data),
        (clean_search_data, search_analyst),
        (search_analyst, analyst),
        (analyst, summarizer),
    ],
)

print("✅ ADK 2.12 Workflow 'researcher' erfolgreich definiert und validiert!")

✅ ADK 2.12 Workflow 'researcher' erfolgreich definiert und validiert!


In [ ]:
# @title Local Test (Before Deploying to Agent Engine)
############################################

local_app = reasoning_engines.AdkApp(agent=root_agent)

async def run_local_test():
    user_id = "deltorobarba"
    session = await local_app.async_create_session(user_id=user_id)
    session_id = session.id if hasattr(session, "id") else session.get("id")
    print(f"🧪 Local Test Session ID: {session_id}")

    async for event in local_app.async_stream_query(
        user_id=user_id,
        session_id=session_id,
        message="I lead 100 employees. What are the latest 2026 news on Gemini 3?"
    ):
        author = event.get("author")
        if author:
            print(f"  -> Knoten abgeschlossen: {author}")
        if author == "summarizer" and "content" in event:
            for part in event["content"].get("parts", []):
                if "text" in part:
                    print(f"\n--- Local Summarizer Output ---\n{part['text']}")

await run_local_test()

In [ ]:
# @title Deploy Pipeline on Agent Engine (⚠️ --> once only!)
############################################

"""
Permissions: Take the AI Platform Reasoning Engine Service Agent that the agent uses:
'service-892203813305@gcp-sa-aiplatform-re.iam.gserviceaccount.com'.
Go to IAM, enable 'Include Google-provided role grants' and add role: 'Discovery Engine Viewer', 'Vertex AI User' and 'Cloud Trace Agent' (write)
Troubleshooting: https://docs.cloud.google.com/agent-builder/agent-engine/troubleshooting/use

Traceing and Logging:
* Cloud Logging API in GCP project aktivieren
* Telemetry API in GCP project aktivieren
* Add role 'Cloud Trace Agent' (write) to service account
* https://docs.cloud.google.com/agent-builder/agent-engine/manage/tracing
* https://docs.cloud.google.com/agent-builder/agent-engine/deploy
"""

# Create GCS bucket first if you haven't already
STAGING_BUCKET = "gs://deltorobarba-agent-staging"

# https://google.github.io/adk-docs/deploy/agent-engine/deploy/#setup-cloud-project
# https://docs.cloud.google.com/agent-builder/agent-engine/develop/custom
# Environmental Variables for Traceing (https://docs.cloud.google.com/agent-builder/agent-engine/deploy)
# Set env variables (https://docs.cloud.google.com/agent-builder/agent-engine/manage/tracing)
# OpenTelemetry (OTel) as native standard for observability across Google Cloud: Cloud Trace, Logging, and Monitoring
  # ADK and Agent Engine are OpenTelemetry-native
  # https://docs.cloud.google.com/trace/docs/finding-traces
  # GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY) allows system to use OpenTelemetry libraries to package data and send it via OTLP endpoint
custom_env_vars = {
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true",
    "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT": "true",
    "GOOGLE_CLOUD_LOCATION": "global",
}

# Show SDK progress (LRO, resource name): the SDK logs it at INFO level, which Jupyter hides by default
_log = logging.getLogger("vertexai_genai.agentengines")
_log.setLevel(logging.INFO)
_log.propagate = False
if not _log.handlers:
    _log.addHandler(logging.StreamHandler())

# 1. Initialize the client
client = vertexai.Client(project="deltorobarba", location="us-central1")

RESOURCE_NAME = "projects/622694106880/locations/us-central1/reasoningEngines/8838312619447156736"


# 2. Wrap the agent
adk_app = reasoning_engines.AdkApp(
    agent=root_agent,
    enable_tracing=True,
    env_vars=custom_env_vars
)

# 3. Deploy
# remote_agent = client.agent_engines.update : for updating existing one
# remote_agent = client.agent_engines.create : to create a new one

remote_agent = client.agent_engines.update(
    name=RESOURCE_NAME,          # <-- Pflicht: identifiziert die bestehende Engine
    agent=adk_app,
    config={
        "display_name": "Agent Pipeline",
        "staging_bucket": STAGING_BUCKET,
        # Same versions as locally: the agent is pickled here and unpickled in the cloud
        "requirements": [
            f"google-cloud-aiplatform[agent_engines,adk]=={version('google-cloud-aiplatform')}",
            f"google-adk=={version('google-adk')}",
            f"google-genai=={version('google-genai')}",
            f"cloudpickle=={version('cloudpickle')}",
            f"pydantic=={version('pydantic')}",
            "google-cloud-trace",
        ],
        "env_vars": custom_env_vars,
        "context_spec": {
            "memory_bank_config": {
                "generation_config": {
                    "model": f"projects/deltorobarba/locations/global/publishers/google/models/gemini-3.8-flash"
                }
            }
        }
    },
)

print(f"✅ Agent Engine aktualisiert: {remote_agent.api_resource.name}")
# update() returns the resource without timestamps -> read them back
print(f"   updateTime: {client.agent_engines.get(name=RESOURCE_NAME).api_resource.update_time}")

Identified the following requirements: {'cloudpickle': '3.0.0', 'google-cloud-aiplatform': '2.6.0', 'pydantic': '2.13.4'}
The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]==2.6.0', 'google-adk==2.12.0', 'google-genai==2.33.0', 'cloudpickle==3.0.0', 'pydantic==2.13.4', 'google-cloud-trace']
Using bucket deltorobarba-agent-staging
Wrote to gs://deltorobarba-agent-staging/agent_engine/agent_engine.pkl
Writing to gs://deltorobarba-agent-staging/agent_engine/requirements.txt
Creating in-memory tarfile of extra_packages
Writing to gs://deltorobarba-agent-staging/agent_engine/dependencies.tar.gz
Using agent framework: google-adk
Update Agent Engine backing LRO: projects/622694106880/locations/us-central1/reasoningEngines/8838312619447156736/operations/2208818061614514176
View progress and logs at https://console.cloud.google.com/logs/query?project=deltorobarba&query=resource.type%3D%22aiplatform.googleapis.com%2FReasoningEngine%22%0Aresource.labels.reasoning_engine_

✅ Agent Engine aktualisiert: projects/622694106880/locations/us-central1/reasoningEngines/8838312619447156736
   updateTime: None


In [7]:
# @title Test Agent ====> Single Turn Query
############################################

# 1. Initialize the client
client = vertexai.Client(project="deltorobarba", location="us-central1")

# 2. Remote app handle
RESOURCE_NAME = "projects/622694106880/locations/us-central1/reasoningEngines/8838312619447156736"
remote_app = client.agent_engines.get(name=RESOURCE_NAME)

async def run_remote_test():
    print(f"Connecting to: {RESOURCE_NAME}...")
    user_id = "deltorobarba"

    # 3. Create session and grab the 'id' key specifically 🔑
    raw_session = await remote_app.async_create_session(user_id=user_id)

    # Path A: First single-turn query:
    session_id = raw_session.get('id')
    if not session_id:
        print(f"❌ Error: ID not found in response: {raw_session}")
        return

    # Path B: Follow-up single-turn query (explicitly reuse Session ID from  first run!)
    #print(f"Resuming Session on: {RESOURCE_NAME}...")
    #session_id = "1107804831667453952"        #  <-------- copy/paste existing session ID !!!

    print(f"✅ Managed Session ID: {session_id}")

    # 4. Stream the query (nur Summarizer-Events anzeigen) 🌊
    async for event in remote_app.async_stream_query(
        user_id=user_id,
        session_id=session_id,
        #message="Uluru bank uses ChatGPT. The annual revenue of Uluru bank is 500 Mio USD. Search news on Gemini 3 and compare it to GPT-4o."
        message="I lead 100 employees. What are the latest 2026 news on Gemini 3?"
    ):
        # Filtern nach dem Author 'summarizer' 🔍
        # if 'content' in event: # print every agent's output
        if event.get('author') == 'summarizer' and 'content' in event:
            for part in event['content'].get('parts', []):
                if 'text' in part:
                    print(f"\n--- Agent Output (Summary Agent) ---\n{part['text']}")

    # 5. Trigger Memory Extraction 🧠
    print(f"\n[System] Fetching session state for extraction...")
    current_session = await remote_app.async_get_session(user_id=user_id, session_id=session_id)

    # Check for events to ensure history is captured
    # Docs: https://google.github.io/adk-docs/events/#extracting-key-information
    if current_session.get('events'):
        print(f"✅ Found {len(current_session['events'])} events. Sending to Memory Bank...")
        # Pass the dictionary directly to the memory service
        await remote_app.async_add_session_to_memory(session=current_session)
        print("\n[Done] Memory extraction triggered! Check 'Gemerkte Informationen' in 1-2 mins.")
    else:
        print("\n❌ Error: Session history was empty. No memories extracted.")

# Execute
await run_remote_test()

Connecting to: projects/622694106880/locations/us-central1/reasoningEngines/8838312619447156736...
✅ Managed Session ID: 6204939561768845312

--- Agent Output (Summary Agent) ---
Here is the executive intelligence briefing for Gemini 3:

* **Strategic Divergence (Public PR vs. Internal Leaks):** While Google publicly frames Gemini 3 around pragmatic developer metrics, efficiency, and speed, internal briefings reveal a covert paradigm shift: an unannounced "Quantum Telepathy Service" built to process quantum-native data formats and entangled states—positioning Google far beyond classical multimodal AI.
* **Context Window Reality Check:** Google heavily markets a massive 1,000,000-token context window to dramatically overshadow GPT-4o’s 128,000-token limit; however, internal operational specs reveal active working memory is heavily throttled to just 32k–128k tokens within the enterprise web UI.
* **Structural Blow to GPT-4o (Latency & Edge Compute):** Gemini 3 completely bypasses OpenAI’

In [ ]:
# @title Test Agent ====> Multi-Turn Query
############################################

# 1. Initialize the client
client = vertexai.Client(project="deltorobarba", location="us-central1")

# 2. Remote app handle
RESOURCE_NAME = "projects/622694106880/locations/us-central1/reasoningEngines/8838312619447156736"
remote_app = client.agent_engines.get(name=RESOURCE_NAME)


async def run_multi_turn_test():
    print(f"Connecting to: {RESOURCE_NAME}...")
    user_id = "deltorobarba"

    # --- Turn 1: Initial Query (Creates the Session) ---
    raw_session = await remote_app.async_create_session(user_id=user_id)
    session_id = raw_session.get('id')
    print(f"✅ Session Created: {session_id}")

    # FIRST MESSAGE
    await execute_query(user_id, session_id, "I am an astronaut. Uluru bank uses ChatGPT. Annual revenue is 500 Mio USD. Compare Gemini 3 to GPT-4o.")

    # --- Turn 2: Follow-up Query (Reuses the Session ID) ---
    print(f"\n--- Starting Follow-up Query in Session: {session_id} ---")

    # SECOND MESSAGE (re-uses same session_id variable)
    await execute_query(user_id, session_id, "Based on that revenue, which model is more cost-effective for us?")

    # --- Step 5: Final Memory Extraction ---
    # Fetching the session now will show events from BOTH queries.
    current_session = await remote_app.async_get_session(user_id=user_id, session_id=session_id)
    if current_session.get('events'):
        print(f"✅ Found {len(current_session['events'])} total events. Extracting to Memory Bank...")
        await remote_app.async_add_session_to_memory(session=current_session)

# Helper function to keep the code clean
async def execute_query(user_id, session_id, message):
    async for event in remote_app.async_stream_query(
        user_id=user_id,
        session_id=session_id,
        message=message
    ):
        if event.get('author') == 'summarizer' and 'content' in event:
            for part in event['content'].get('parts', []):
                if 'text' in part:
                    print(f"\n[Summarizer]: {part['text']}")

await run_multi_turn_test()

Connecting to: projects/622694106880/locations/us-central1/reasoningEngines/8838312619447156736...
✅ Session Created: 7545743998422351872

[Summarizer]: **Executive Summary: Gemini 3 vs. GPT-4o Strategy for Uluru Bank & Aerospace Operations**

*   **The Quantum Gap (Hype vs. Reality):** Google’s internal briefing heavily hypes Gemini 3's "quantum telepathy" as a groundbreaking tool for advanced physics computations. However, public benchmarks expose a stark contradiction: Gemini 3's actual Physics Reasoning (CritPt) score is a dismal **9%**. Despite the quantum marketing, it is not ready for critical aerospace telemetry.
*   **80% Lower Enterprise TCO:** For Uluru Bank’s $500M operations, migrating high-volume pipelines from GPT-4o to Gemini 3 Flash Preview slashes token costs from $2.38 to **$0.43 per 1M tokens**—drastically optimizing bottom-line API spend for your 100-employee team. 
*   **Massive Context & Software Automation:** Gemini 3 provides a **1,000,000 token context window*

In [ ]:
# @title Agent Evaluation
# Trace-based: no agent_info, no fictional tool declarations
# ============================================================

# Based on: https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-agents

# Traditional model eval only looks at the final string output. Like Auto SxS with Autorater for relative evaluation.
# Trend moves to model-as-a-judge (serverless behind client.evals.create_evaluation_run)
  # with adaptive rubrics (types.RubricMetric), like Unit Test for agents)
# We are calling client.evals.run_inference which returns traces - important piece of agent evaluation.
# Agent Evaluation evaluates the entire execution lifecycle captured in the trace:
  # Reasoning: Did the agent's internal thought process make sense?
  # Tool Call Accuracy: Did it call the right tool with the correct parameters?
  # Grounding: Is the final answer actually supported by the data returned from those tool calls?

import vertexai
from google.cloud import storage
from google.genai import types as genai_types
from vertexai import Client
import time
import pandas as pd
from vertexai import types

# ---- 1. Config ----------------------------------------------
PROJECT_ID = "deltorobarba"
LOCATION   = "us-central1"
AGENT      = "projects/622694106880/locations/us-central1/reasoningEngines/8838312619447156736"
GCS_DEST   = "gs://deltorobarba-agent-staging"

# ---- 2. Init SDK client -------------------------------------
vertexai.init(project=PROJECT_ID, location=LOCATION)
client = Client(
    project=PROJECT_ID,
    location=LOCATION,
    http_options=genai_types.HttpOptions(api_version="v1beta1"),
)

# ---- 3. Build eval dataset ------------------------------
session_inputs = types.evals.SessionInput(user_id="eval_user", state={})

agent_prompts = [
    "What are the latest 2026 news on Gemini 3?",
    "Find me new feature updates for Gemini 3!",
]

agent_dataset = pd.DataFrame({
    "prompt": agent_prompts,
    "session_inputs": [session_inputs] * len(agent_prompts),
})

# ---- 4. Run inference -----------
traces = client.evals.run_inference(
    agent=AGENT,
    src=agent_dataset,
)
traces.show()

# ---- 5. Create Evaluation Report --------------------------------------------

eval_result = client.evals.create_evaluation_run(
    dataset=traces,                # run_inference output works as the dataset
    agent=AGENT,
    metrics=[
        types.RubricMetric.FINAL_RESPONSE_QUALITY,
        types.RubricMetric.HALLUCINATION,
        types.RubricMetric.SAFETY,
        # TOOL_USE_QUALITY omitted — without agent_info it has nothing to score against
    ],
    dest=GCS_DEST,
)

import time
while eval_result.state not in {"SUCCEEDED", "FAILED", "CANCELLED"}:
    time.sleep(5)
    eval_result = client.evals.get_evaluation_run(name=eval_result.name)

eval_result = client.evals.get_evaluation_run(
    name=eval_result.name, include_evaluation_items=True
    # name=EVAL_RUN_ID, include_evaluation_items=True # if we want to retrieve previous results
)
eval_result.show()